In [1]:
def itembased(input_item,ratings,items,similar_item_userid_count,threshold):
    n_users=ratings.user_id.unique().shape[0]
    n_items=ratings.movie_id.unique().shape[0]
    
    ## Creating Pivot table
    datama=ratings.pivot_table(index='user_id',columns='movie_id',values='rating')
    
    ##Replace Nan with 0
    data_matrix=datama.replace(np.nan,0)
    
    ##Finding Similarity between users & Similarity between items
    from sklearn.metrics.pairwise import pairwise_distances
    user_similarity=pairwise_distances(data_matrix,metric='cosine')
    item_similarity=pairwise_distances(data_matrix.T,metric='cosine')
    
    # Calculate Score value using formula
    def predict(ratings,similarity,type='user'):
        if(type == 'user'):
            mean_user_rating=ratings.mean(axis=1)   ## the average (mean) value across each row is calculated
            #We use np.newaxis so that mean_user_rating hassame format as ratings
            rating_diff=(ratings-mean_user_rating.to_numpy()[:,np.newaxis])
            pred=mean_user_rating.to_numpy()[:,np.newaxis] + similarity.dot(rating_diff)/np.array([np.abs(similarity).sum(axis=1)]).T
        elif type == 'item':
            pred = ratings.dot(similarity)/np.array([np.abs(similarity).sum(axis=1)])
        return pred
    
    #Prediction table
    user_prediction=predict(data_matrix,user_similarity,type='user')
    item_prediction=predict(data_matrix,item_similarity,type='item')

    ## To find the similarity between input item and other items
    ## convert item_similarity into table

    item_sim_table=pd.DataFrame(item_similarity)

    ## Find similar user for input user using cosine table
    similar_input_item=item_sim_table[input_item].sort_values(ascending=True).head(similar_item_userid_count).index

    #Convert similar_input_item to list
    similar_item_input=list(similar_input_item)

    ## Creating a list of user_id for similar item
    similar_item_userid_list=[]
    for sim_item in similar_input_item:
        sim=list(ratings[ratings['movie_id']==sim_item]['user_id'])
        similar_item_userid_list.append(sim)

    # Convert all the list into single list
    import itertools
    similar_item_userid_single_list=list(itertools.chain.from_iterable(similar_item_userid_list))

    #Unique user id from the list
    Unique_userid_similar_item=set(similar_item_userid_single_list)
    #Input item watched user list
    input_item_watched_userid=list(ratings[ratings['movie_id']==input_item]['user_id'].values)

    ## Create a list of userid for item input
    recomm=[]
    for per_id in Unique_userid_similar_item:
        if(per_id in input_item_watched_userid):
            pass
        else:
           recomm.append(per_id)

    ##Filterd userid have to be checked with prediction table for ratings
    item_pred=pd.DataFrame(item_prediction)

    
    ## from the recomm list select the users who likes the movieid most using prediction table
    highest_Rated=[]
    input_item_pre=pd.DataFrame(item_pred[input_item])
    input_item_pred=input_item_pre.T
    for re in recomm:
        value=input_item_pred[re].values
        if(value>=threshold):
            highest_Rated.append(re)
    

    return highest_Rated

In [3]:
import pandas as pd
import numpy as np

input_item=5
similar_item_userid_count=5
threshold=1

r_cols=['user_id','movie_id','rating','unix_timestamp']
ratings=pd.read_csv("ml-100k/u.data",sep='\t', names=r_cols)  ## can add encoding='latin-1'


# Get the respective movie names for movie id

i_cols = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 'Adventure',
'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


items = pd.read_csv('ml-100k/u.item', sep='|', names=i_cols,encoding='latin-1')

final_recomm_user_list=itembased(input_item,ratings,items,similar_item_userid_count,threshold)

print("Final_recommended_users_list:\n", final_recomm_user_list)

Final_recommended_users_list:
 [416, 450]
